In [ ]:
VOCABULAR = [c for c in "ABCDEFGHIJ"]
CONTEXT_LENGTH = 20
EMBEDDING_DIM = 16
NUM_HEADS = 10
MODEL_NAME = "demo_model"
EXPERIMENT_NAME = "demo_experiment"
ACT_SITES = ["blocks.0.hook_resid_post", "blocks.1.hook_resid_post"]

In [ ]:
import copy_transformer.training
import transformer_lens
import infra.dataset_configs as dataset_configs
import copy_transformer.tokenizer
from pathlib import Path

training_dataset_config = dataset_configs.PureRepeatingPatternConfig(
    vocabulary=VOCABULAR,
    context_length=CONTEXT_LENGTH,
    max_pattern_length=10,
    iterable=False,
    length=1_000,
)

training_config = copy_transformer.training.TrainingConfig(
    model_name=MODEL_NAME,
    epochs=10,
    dataset_config=training_dataset_config,
    validation_dataset_config=training_dataset_config,
)

tokenizer = copy_transformer.tokenizer.SingleCharTokenizer(
    alphabet=VOCABULAR,
    bos_token=">",
    eos_token="<",
    unk_token="?",
    pad_token="_",
)

model_config = transformer_lens.HookedTransformerConfig(
    d_model=EMBEDDING_DIM,
    n_heads=NUM_HEADS,
    d_head=EMBEDDING_DIM // NUM_HEADS,
    n_layers=2,
    n_ctx=CONTEXT_LENGTH,
    attn_only=True,
    d_vocab=tokenizer.vocab_size,
)

model = copy_transformer.training.train_transformer(
    config=training_config,
    model_config=model_config,
    tokenizer=tokenizer,
)

In [ ]:
import subspace_partition.subspace_partition

subspace_partition_dataset_config = dataset_configs.PureRepeatingPatternConfig(
    vocabulary=VOCABULAR,
    context_length=CONTEXT_LENGTH,
    max_pattern_length=10,
    iterable=True,
    length="infinite",
)

subspace_partition_config = (
    subspace_partition.subspace_partition.SubspacePartitionConfig(
        exp_name=EXPERIMENT_NAME,
        model_name=MODEL_NAME,
        dataset_config=subspace_partition_dataset_config,
        act_sites=ACT_SITES,
        max_steps=204,
        merge_start=200,
        merge_interval=200,
        unit_size=8,  # Must divide EMBEDDING_DIM (16) evenly
    )
)

subspace_partition.subspace_partition.run_subspace_partition(
    cfg=subspace_partition_config
)

In [ ]:
import subspace_partition.preimage.cache_act

cached_act_dataset_config = dataset_configs.PureRepeatingPatternConfig(
    vocabulary=VOCABULAR,
    context_length=CONTEXT_LENGTH,
    max_pattern_length=10,
    iterable=True,
    length=10_000,
)

subspace_partition.preimage.cache_act.run_cache_act(
    model_name=MODEL_NAME,
    dataset_config=cached_act_dataset_config,
    act_sites=ACT_SITES,
)

In [ ]:
import subspace_partition.preimage.build_index

subspace_partition.preimage.build_index.run_build_index(experiment_name=EXPERIMENT_NAME)

In [ ]:
import os
import infra

! "{infra.PROJECT_ROOT}/start_streamlit_app.sh" "{infra.OUT_DIR}/index/index-{EXPERIMENT_NAME}-cosine"